In [ ]:

---

### 2. Complete Jupyter Notebook

Create the file `notebooks/01_eda_and_modeling.ipynb` and paste the cells below one by one.

**Cell 1 – Imports & Paths**
```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import (roc_auc_score, accuracy_score, classification_report,
                             precision_score, recall_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

%matplotlib inline
sns.set(style="whitegrid")

# Paths relative to the notebook location
ROOT = Path().resolve().parent
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "model"
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", ROOT)

In [ ]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login",
    "count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df_train = pd.read_csv(DATA_DIR / "KDDTrain+.txt", names=columns)
df_test  = pd.read_csv(DATA_DIR / "KDDTest+.txt",  names=columns)

print("Train shape:", df_train.shape)
print("Test  shape:", df_test.shape)
df_train.head()

In [ ]:
df_train["target"] = (df_train["label"] != "normal").astype(int)
df_test["target"]  = (df_test["label"]  != "normal").astype(int)

print("Train target distribution:")
display(df_train["target"].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df_train, x="target", ax=axes[0])
axes[0].set_title("Target Distribution (0 = Normal, 1 = Attack)")

sns.countplot(data=df_train, x="protocol_type", hue="target", ax=axes[1])
axes[1].set_title("Protocol Type vs Target")
plt.tight_layout()
plt.show()

In [ ]:
df_train = df_train.drop(columns=["label", "difficulty"])
df_test  = df_test.drop(columns=["label", "difficulty"])

categorical = ["protocol_type", "service", "flag"]
numerical = [c for c in df_train.columns if c not in categorical + ["target"]]

def prepare(df):
    df = df.copy()
    df[categorical] = df[categorical].astype(str)
    return df

df_train = prepare(df_train)
df_test  = prepare(df_test)

In [ ]:
df_full_train, df_val = train_test_split(
    df_train, test_size=0.2, random_state=42, stratify=df_train["target"]
)

y_train = df_full_train.target.values
y_val   = df_val.target.values
y_test  = df_test.target.values

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(df_full_train[categorical + numerical].to_dict(orient="records"))
X_val   = dv.transform(df_val[categorical + numerical].to_dict(orient="records"))
X_test  = dv.transform(df_test[categorical + numerical].to_dict(orient="records"))

print("X_train shape:", X_train.shape)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=120, max_depth=6, learning_rate=0.1,
                             random_state=42, n_jobs=-1, eval_metric="logloss")
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    
    results[name] = {
        "model": model,
        "auc": roc_auc_score(y_val, y_pred),
        "accuracy": accuracy_score(y_val, y_pred >= 0.5),
        "f1": f1_score(y_val, y_pred >= 0.5)
    }
    
    print(f"{name:20} | AUC: {results[name]['auc']:.4f} | F1: {results[name]['f1']:.4f}")

In [ ]:
best_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_name]["model"]
print("Best model:", best_name)

y_pred_test = best_model.predict_proba(X_test)[:, 1]
print("\nTest AUC:", roc_auc_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test >= 0.5))

In [ ]:
joblib.dump({
    "model": best_model,
    "dv": dv,
    "categorical": categorical,
    "numerical": numerical
}, MODEL_DIR / "model.bin")

print("Model saved successfully to:", MODEL_DIR / "model.bin")